# Suite VLG — The Verilog language

Verilog and SystemVerilog as one tree language (`mbse.Programs.Verilog.Syntax`): its kinds and categories,
organized as IEEE 1800's grammar is, the identifiers it accepts, and where each kind and feature exists in the two
families, which the standards check.

In [ ]:
from mbse.Programs.Framework import Syntax as F
from mbse.Programs.Verilog import SystemVerilog2017, SystemVerilog2023, Syntax as S, Verilog2005, VerilogStandard

L = S.LANGUAGE
V1995, V2001 = VerilogStandard(1995, "Verilog"), VerilogStandard(2001, "Verilog")
SV2005, SV2009, SV2012 = VerilogStandard(2005), VerilogStandard(2009), VerilogStandard(2012)


def name(spelling):
    return S.Identifier(spelling=spelling)


def ref(spelling):
    return S.NameExpression(name=name(spelling))


def number(spelling="1"):
    return S.IntegerLiteral(spelling=spelling)


def unit(*items):
    return S.SourceText(items=list(items))


def module(*items, **properties):
    return S.ModuleDeclaration(keyword="module", name=name("m"), items=list(items), **properties)

## VLG-01 · Every kind belongs to one category, and the grammar lists them

In [ ]:
kinds = L.kinds()
assert len(kinds) == len(S.KINDS) == 127 and list(kinds) == [k.__name__ for k in S.KINDS]
assert sorted(L.categories()) == sorted(["Name", "Item", "Port", "DataType", "Dimension", "Statement", "Connection",
                                        "TimingControl", "Expression", "Literal", "Range", "Directive"])
assert issubclass(S.IntegerLiteral, S.Literal) and issubclass(S.Literal, S.Expression)
assert all(k.NAME == f"Programs.Verilog.{k.KIND}" for k in S.KINDS)
grammar = L.grammar()
assert grammar["base"] == {"Verilog": 1995, "SystemVerilog": 2005} and len(grammar["kinds"]) == 127
assert S.sv() == {"SystemVerilog": 2005} and S.sv(2009) == {"SystemVerilog": 2009}
assert S.verilog(2001) == {"Verilog": 2001, "SystemVerilog": 2005}
assert S.Comment(text=" x").block is False and S.SourceText().items == []
print("ok")

## VLG-02 · Identifiers are simple or escaped, and system names start with `$`

In [ ]:
assert L.validate(name("x_1$")) == [] and L.validate(name("\\bus+index")) == [] and L.validate(name("")) == []
for bad in ["1x", "$x", "a b", "Größe", "\\", "\\a b"]:
    assert L.validate(name(bad)) == [f"{bad!r} is not an identifier"], bad
assert L.validate(S.Identifier(spelling=3)) == ["Identifier.spelling must be a str, got int"]
assert L.validate(S.SystemCall(name="$clog2")) == [] and L.validate(S.SystemCall(name="")) == []
for bad in ["clog2", "$", "$a b", "$é"]:
    assert L.validate(S.SystemCall(name=bad)) == [f"{bad!r} is not a system task or function name"], bad
assert L.validate(S.SystemCall(name=1)) == ["SystemCall.name must be a str, got int"]
print("ok")

## VLG-03 · Kinds and features exist by family and year: Verilog has none of SystemVerilog's own

In [ ]:
labels = [Verilog2005.STANDARD.name(), SystemVerilog2017.STANDARD.name(), SystemVerilog2023.STANDARD.name(), V1995.name()]
assert labels == ["Verilog-2005", "SystemVerilog-2017", "SystemVerilog-2023", "Verilog-1995"]
package = S.PackageDeclaration(name=name("p"))
assert Verilog2005.check(package) == ["PackageDeclaration is not Verilog"] and SV2005.check(package) == []
ansi = S.AnsiPort(direction="input", name=name("a"))
assert V1995.check(ansi) == ["AnsiPort needs Verilog-2001"] and V2001.check(ansi) == []
logic = S.IntegerVectorType(keyword="logic")
assert Verilog2005.check(logic) == ["IntegerVectorType.keyword 'logic' is not Verilog"]
assert Verilog2005.check(S.IntegerVectorType(keyword="reg", signing="signed")) == []
assert V1995.check(S.IntegerVectorType(keyword="reg", signing="signed")) == ["IntegerVectorType.signing 'signed' needs Verilog-2001"]
unique0 = S.IfStatement(qualifier="unique0", condition=ref("a"), consequence=S.NullStatement())
assert SV2005.check(unique0) == ["IfStatement.qualifier 'unique0' needs SystemVerilog-2009"] and SV2009.check(unique0) == []
final = S.ImmediateAssertion(keyword="assert", deferral="final", expression=ref("a"))
assert SV2009.check(final) == ["ImmediateAssertion.deferral 'final' needs SystemVerilog-2012"] and SV2012.check(final) == []
star = S.EventControl()
assert V1995.check(star) == ["EventControl * needs Verilog-2001"] and V2001.check(star) == []
fork = S.DisableStatement()
assert Verilog2005.check(fork) == ["DisableStatement fork is not Verilog"]
assert Verilog2005.check(S.DisableStatement(target=ref("b"))) == []
declared = S.ForStatement(initializers=[S.VariableDeclaration(type=S.IntegerAtomType(keyword="integer"), declarators=[
    S.VariableDeclarator(name=name("i"), value=number())])], body=S.NullStatement())
assert Verilog2005.check(declared) == ["ForStatement.initializers declarations is not Verilog"]
two = S.ForStatement(initializers=[S.AssignmentExpression(target=ref("i"), operator="=", value=number()),
                                   S.AssignmentExpression(target=ref("j"), operator="=", value=number())], body=S.NullStatement())
assert Verilog2005.check(two) == ["ForStatement.initializers declarations is not Verilog"] and SV2005.check(two) == []
one = S.ForStatement(initializers=[S.AssignmentExpression(target=ref("i"), operator="=", value=number())], body=S.NullStatement())
assert Verilog2005.check(one) == []
labeled = module(labeled=True)
assert Verilog2005.check(labeled) == ["ModuleDeclaration.labeled is not Verilog"]
assert V1995.check(S.BinaryExpression(left=number(), operator="**", right=number())) == [
    "BinaryExpression.operator '**' needs Verilog-2001"]
assert SV2005.check(S.BinaryExpression(left=number(), operator="<->", right=number())) == [
    "BinaryExpression.operator '<->' needs SystemVerilog-2009"]
interface_class = S.ClassDeclaration(interface=True, name=name("i"))
assert Verilog2005.check(interface_class) == ["ClassDeclaration is not Verilog", "ClassDeclaration.interface is not Verilog"]
assert SV2009.check(interface_class) == ["ClassDeclaration.interface needs SystemVerilog-2012"]
assert SV2009.check(S.ForwardTypedefDeclaration(keyword="interface class", name=name("i"))) == [
    "ForwardTypedefDeclaration.keyword 'interface class' needs SystemVerilog-2012"]
assert Verilog2005.check(S.VariableDeclaration(random="rand", declarators=[S.VariableDeclarator(name=name("r"))])) == [
    "VariableDeclaration.random is not Verilog"]
assert Verilog2005.check(S.NewExpression()) == ["NewExpression is not Verilog"]
print("ok")

## VLG-04 · A standard checks a whole tree, by path

In [ ]:
tree = unit(module(S.AlwaysConstruct(keyword="always_ff", body=S.NullStatement()),
                   S.ContinuousAssign(assignments=[S.AssignmentExpression(target=ref("a"), operator="+=", value=number())])))
assert Verilog2005.check(tree) == ["items[0].items[0]: AlwaysConstruct.keyword 'always_ff' is not Verilog",
                                   "items[0].items[1].assignments[0]: AssignmentExpression.operator '+=' is not Verilog"]
assert SystemVerilog2023.check(tree) == []
print("ok")

## VLG-05 · Classes and methods check their own shape

In [ ]:
def method(kind, **properties):
    return kind(name=name("f"), **properties)


for kind in (S.FunctionDeclaration, S.TaskDeclaration):
    assert L.validate(method(kind, virtual=True, pure=True)) == [] and L.validate(method(kind, extern=True)) == []
    assert L.validate(method(kind, pure=True)) == [f"a pure {kind.__name__} is virtual"]
    assert L.validate(method(kind, extern=True, body=[S.NullStatement()])) == [f"an extern or pure {kind.__name__} has no body"]
    assert L.validate(method(kind, virtual=True, pure=True, labeled=True)) == [f"an extern or pure {kind.__name__} has no body"]
base = S.NamedType(name=name("b"))
assert L.validate(S.ClassDeclaration(name=name("c"), base=base, arguments=[number()])) == []
assert L.validate(S.ClassDeclaration(interface=True, name=name("i"), interfaces=[S.NamedType(name=name("j"))])) == []
assert L.validate(S.ClassDeclaration(interface=True, name=name("i"), base=S.NamedType(name=name("b")))) == [
    "an interface ClassDeclaration has no virtual, base or arguments"]
assert L.validate(S.ClassDeclaration(interface=True, virtual=True, name=name("i"))) == [
    "an interface ClassDeclaration has no virtual, base or arguments"]
assert L.validate(S.ClassDeclaration(name=name("c"), arguments=[number()])) == ["a ClassDeclaration with arguments has a base"]
print("ok")